# 火灾疏散项目 — Colab 训练脚本 (网页版)
**使用方法**
1. 打开 https://colab.research.google.com/ → 文件 → 上传笔记本 → 选这个文件
2. 菜单栏: 修改 → 笔记本设置 → 硬件加速器选 **T4 GPU** → 保存
3. 按 **Ctrl+F9** 全部运行，等 8 分钟
4. 最后一步浏览器自动下载 best_model.pth 到本地

In [ ]:
# [1] 克隆代码 + 数据集
%cd /content
!rm -rf LonlyStydue Social-STGCNN
!git clone -q https://github.com/confidentismylife/LonlyStydue.git
!git clone -q https://github.com/abduallahmohamed/Social-STGCNN.git
!pip install -q torch numpy pandas matplotlib scipy scikit-learn tqdm

import os
data_dir = '/content/Social-STGCNN/datasets'
txt_files = []
for root, dirs, files in os.walk(data_dir):
    for f in files:
        if f.endswith('.txt'):
            txt_files.append(os.path.join(root, f))

print(f'数据文件: {len(txt_files)} 个')
if txt_files:
    with open(txt_files[0], 'r') as fh:
        print(f'样本格式: {repr(fh.readline().strip())}')
else:
    raise RuntimeError('数据克隆失败，请重新运行本单元格')

In [ ]:
# [2] 检查 GPU
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'型号: {torch.cuda.get_device_name(0)}')
else:
    print('警告: 没有 GPU！去菜单 修改→笔记本设置 选 T4')

In [ ]:
# [3] 加载数据
import sys
sys.path.insert(0, '/content/LonlyStydue')

import numpy as np
from collections import defaultdict
OBS_LEN, PRED_LEN = 8, 12

def load_file(fp):
    data = defaultdict(list)
    with open(fp, 'r') as f:
        for line in f:
            parts = line.strip().replace('\t', ' ').split()
            if len(parts) < 4:
                continue
            try:
                frame = int(float(parts[0]))
                pid = int(float(parts[1]))
                x, y = float(parts[2]), float(parts[3])
                data[pid].append((frame, x, y))
            except:
                continue
    return data

def extract(data):
    olist, plist = [], []
    for pid, traj in data.items():
        traj.sort(key=lambda t: t[0])
        for i in range(len(traj) - OBS_LEN - PRED_LEN + 1):
            seg = traj[i:i+OBS_LEN+PRED_LEN]
            olist.append(np.array([[t[1],t[2]] for t in seg[:OBS_LEN]], dtype=np.float32))
            plist.append(np.array([[t[1],t[2]] for t in seg[OBS_LEN:]], dtype=np.float32))
    return olist, plist

all_obs, all_preds = [], []
for fp in txt_files:
    raw = load_file(fp)
    obs, preds = extract(raw)
    all_obs.extend(obs)
    all_preds.extend(preds)

all_obs = np.array(all_obs)
all_preds = np.array(all_preds)
print(f'轨迹总数: {len(all_obs):,}  |  obs shape: {all_obs.shape}  |  pred shape: {all_preds.shape}')

In [ ]:
# [4] 归一化 + 划分训练/验证集
origin = all_obs[:, -1:, :]
all_obs_n = all_obs - origin
all_preds_n = all_preds - origin

np.random.seed(42)
idx = np.random.permutation(len(all_obs_n))
n_train = int(len(idx) * 0.8)
train_obs = all_obs_n[idx[:n_train]]
train_preds = all_preds_n[idx[:n_train]]
val_obs = all_obs_n[idx[n_train:]]
val_preds = all_preds_n[idx[n_train:]]
print(f'训练集: {len(train_obs):,} 条  |  验证集: {len(val_obs):,} 条')

In [ ]:
# [5] 训练
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from src.model import SimpleLSTM
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TD(Dataset):
    def __init__(self, o, p):
        self.o = torch.from_numpy(o).float()
        self.p = torch.from_numpy(p).float()
    def __len__(self): return len(self.o)
    def __getitem__(self, i): return self.o[i], self.p[i]

train_loader = DataLoader(TD(train_obs, train_preds), batch_size=128, shuffle=True)
val_loader   = DataLoader(TD(val_obs, val_preds), batch_size=128, shuffle=False)

model = SimpleLSTM(hidden_dim=64, pred_len=PRED_LEN).to(device)
print(f'模型参数: {sum(p.numel() for p in model.parameters()):,}  |  设备: {device}')

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10)
criterion = nn.MSELoss()

best_ade = float('inf')
for epoch in range(1, 101):
    t0 = time.time()

    # 训练
    model.train()
    train_loss = 0.0
    for obs, preds in train_loader:
        obs, preds = obs.to(device), preds.to(device)
        optimizer.zero_grad()
        loss = criterion(model(obs), preds)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * obs.size(0)
    train_loss /= len(train_loader.dataset)

    # 验证
    model.eval()
    ade_sum = fde_sum = count = 0.0
    with torch.no_grad():
        for obs, preds in val_loader:
            obs, preds = obs.to(device), preds.to(device)
            diff = model(obs) - preds
            ade_sum += torch.norm(diff, dim=2).mean(1).sum().item()
            fde_sum += torch.norm(diff[:, -1, :], dim=1).sum().item()
            count += obs.size(0)
    val_ade = ade_sum / count
    val_fde = fde_sum / count
    scheduler.step(val_ade)

    if val_ade < best_ade:
        best_ade = val_ade
        os.makedirs('checkpoints', exist_ok=True)
        torch.save(model.state_dict(), 'checkpoints/best_model.pth')

    if epoch <= 3 or epoch % 20 == 0:
        print(f'Epoch {epoch:3d}/100 | Loss: {train_loss:.4f} | '
              f'ADE: {val_ade:.3f}m | FDE: {val_fde:.3f}m | {time.time()-t0:.1f}s')

print(f'\n训练完成! 最佳 ADE: {best_ade:.3f}m  (匀速基线=0.48m)')

In [ ]:
# [6] 浏览器下载模型文件
from google.colab import files
files.download('checkpoints/best_model.pth')
print('下载完成后，把 best_model.pth 放到 Huo-zai/checkpoints/ 目录下')

In [ ]:
# [7] 可视化 — 看 3 条预测结果
import matplotlib.pyplot as plt
model.eval()
sample_ids = np.random.choice(len(val_obs), 3, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, idx in enumerate(sample_ids):
    oi = torch.from_numpy(val_obs[idx]).float().unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(oi).squeeze(0).cpu().numpy()
    oi = oi.squeeze(0).cpu().numpy()
    gt = val_preds[idx]

    ax = axes[i]
    ax.plot(oi[:, 0], oi[:, 1], 'b-o', label='Observed')
    ax.plot(pred[:, 0], pred[:, 1], 'r--o', label='Predicted')
    ax.plot(gt[:, 0], gt[:, 1], 'g-o', alpha=0.5, label='Ground Truth')
    ax.set_title(f'Sample {i+1}')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('colab_results.png', dpi=100)
plt.show()
print('蓝色=观测 8 帧 | 红色=模型预测 12 帧 | 绿色=真实值')